In [1]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

Python executable: /Users/dafyddjones/.pyenv/versions/3.12.9/envs/lewagon_current/bin/python
Working directory: /Users/dafyddjones/projects/finance-kpi-reporting-pipeline/notebooks


In [2]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DATA_DIR)
print("Raw data folder exists:", RAW_DATA_DIR.is_dir())

Project root: /Users/dafyddjones/projects/finance-kpi-reporting-pipeline
Raw data folder: /Users/dafyddjones/projects/finance-kpi-reporting-pipeline/data/raw
Raw data folder exists: True


In [3]:
import requests

PUBLICATION_URL = (
    "https://www.gov.uk/government/publications/"
    "dft-spending-over-25000-january-2026"
)

response = requests.get(PUBLICATION_URL, timeout=30)
response.raise_for_status()

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))

Status code: 200
Content type: text/html; charset=utf-8


In [4]:
import re
from html import unescape
from urllib.parse import urljoin

csv_links = re.findall(
    r'href="([^"]+\.csv(?:\?[^"]*)?)"',
    response.text,
    flags=re.IGNORECASE
)

csv_urls = sorted({
    urljoin(PUBLICATION_URL, unescape(link))
    for link in csv_links
})

print("CSV links found:", len(csv_urls))

for url in csv_urls:
    print(url)

CSV links found: 2
https://assets.publishing.service.gov.uk/media/6a06eb97ee62840dba48a32a/dft-spending-25k-jan-26.csv
https://www.gov.uk/csv-preview/6a06eb97ee62840dba48a32a/dft-spending-25k-jan-26.csv


In [5]:
from urllib.parse import urlparse

download_urls = [
    url for url in csv_urls
    if urlparse(url).hostname == "assets.publishing.service.gov.uk"
]

if len(download_urls) != 1:
    raise ValueError("Expected exactly one CSV download link.")

CSV_URL = download_urls[0]

print("Selected download:", CSV_URL)

Selected download: https://assets.publishing.service.gov.uk/media/6a06eb97ee62840dba48a32a/dft-spending-25k-jan-26.csv


In [6]:
csv_response = requests.get(CSV_URL, timeout=30)
csv_response.raise_for_status()

file_path = RAW_DATA_DIR / "dft_spending_2026_01.csv"
file_path.write_bytes(csv_response.content)

print("Saved to:", file_path)
print("File size:", file_path.stat().st_size, "bytes")

Saved to: /Users/dafyddjones/projects/finance-kpi-reporting-pipeline/data/raw/dft_spending_2026_01.csv
File size: 455096 bytes


In [7]:
raw_bytes = file_path.read_bytes()

try:
    csv_text = raw_bytes.decode("utf-8-sig")
    csv_encoding = "utf-8-sig"
except UnicodeDecodeError:
    csv_text = raw_bytes.decode("cp1252")
    csv_encoding = "cp1252"

print("Encoding used:", csv_encoding)
print("\nFirst five lines:")

for line in csv_text.splitlines()[:5]:
    print(line)

Encoding used: cp1252

First five lines:
Department Family,Entity,Date,Expense Type,Expense Area,Supplier,Transaction No,Item Text, £ ,Postal Code
Department for Transport,Department for Transport,06/01/2026,Grt Aid to NDPBs,DG Public Transport and Local Group,Network Rail,2000026734,#,"£423,374,000.00",SE1 8SW
Department for Transport,Department for Transport,30/01/2026,Grt Aid to NDPBs,DG Road Transport Group,National Highways Limited,2000029318,Grant in Aid NH for February 2026,"£292,000,000.00",B1 1RN
Department for Transport,High Speed Two (HS2) Limited,14/01/2026,AUC - Phase 1,AUC - 000000,BALFOUR BEATTY VINCI JV - HS2 (N2),#,MWCC BBV IPT Stage 2,"£238,905,785.81",#
Department for Transport,High Speed Two (HS2) Limited,14/01/2026,AUC - Phase 1,AUC - 000000,EIFFAGE KIER FERROVIAL BAM JOINT VENTURE,#,C23 MWCC Stage 2 - EKFB IPT,"£155,310,272.12",#


In [8]:
import pandas as pd

df_raw = pd.read_csv(
    file_path,
    encoding=csv_encoding,
    dtype=str
)

print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

display(df_raw.head())

Rows: 2426
Columns: 10


,Department Family,Entity,Date,Expense Type,Expense Area,Supplier,Transaction No,Item Text,£,Postal Code
0,Department for Transport,Department for Transport,06/01/2026,Grt Aid to NDPBs,DG Public Transport and Local Group,Network Rail,2000026734,#,"£423,374,000.00",SE1 8SW
1,Department for Transport,Department for Transport,30/01/2026,Grt Aid to NDPBs,DG Road Transport Group,National Highways Limited,2000029318,Grant in Aid NH for February 2026,"£292,000,000.00",B1 1RN
2,Department for Transport,High Speed Two (HS2) Limited,14/01/2026,AUC - Phase 1,AUC - 000000,BALFOUR BEATTY VINCI JV - HS2 (N2),#,MWCC BBV IPT Stage 2,"£238,905,785.81",#
3,Department for Transport,High Speed Two (HS2) Limited,14/01/2026,AUC - Phase 1,AUC - 000000,EIFFAGE KIER FERROVIAL BAM JOINT VENTURE,#,C23 MWCC Stage 2 - EKFB IPT,"£155,310,272.12",#
4,Department for Transport,Department for Transport,29/01/2026,TOCOpCosts(Pub),DG Rail Services Group,GOVIA THAMESLINK RAILWAY LIMITED,2000029230,2612 Wk1 GTR_NRC_PCP_Opex_(ICP) GOVIA THAMESLI...,"£153,050,802.26",EC3R 8AJ


In [9]:
df = df_raw.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

df = df.rename(columns={"£": "amount_gbp"})

print(df.columns.tolist())

['department_family', 'entity', 'date', 'expense_type', 'expense_area', 'supplier', 'transaction_no', 'item_text', 'amount_gbp', 'postal_code']


In [10]:
df["entity"] = df["entity"].str.strip()

entity_counts = df["entity"].value_counts(dropna=False)

display(entity_counts)
print("Total rows:", entity_counts.sum())

entity
National Highways                    1181
Department for Transport              659
High Speed Two (HS2) Limited          176
Driver & Vehicle Licensing Agency     111
Maritime and Coastguard Agency         97
DVSA                                   92
British Transport Police               53
EWR                                    49
VCA                                     3
Active Travel England                   3
Transport Focus                         2
Name: count, dtype: int64

Total rows: 2426


In [11]:
df_dft = df.loc[
    df["entity"] == "Department for Transport"
].copy()

print("Central DfT rows:", len(df_dft))
print("Entities retained:", df_dft["entity"].unique())

Central DfT rows: 659
Entities retained: <ArrowStringArray>
['Department for Transport']
Length: 1, dtype: str


In [12]:
df_dft["payment_date"] = pd.to_datetime(
    df_dft["date"].str.strip(),
    format="%d/%m/%Y",
    errors="coerce"
)

print("Invalid or missing dates:", df_dft["payment_date"].isna().sum())
print("Earliest date:", df_dft["payment_date"].min())
print("Latest date:", df_dft["payment_date"].max())

Invalid or missing dates: 0
Earliest date: 2026-01-02 00:00:00
Latest date: 2026-01-30 00:00:00


In [13]:
amount_text = (
    df_dft["amount_gbp"]
    .str.strip()
    .str.replace("£", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace(r"^\((.*)\)$", r"-\1", regex=True)
)

df_dft["payment_amount_gbp"] = pd.to_numeric(
    amount_text,
    errors="coerce"
)

print(
    "Invalid or missing amounts:",
    df_dft["payment_amount_gbp"].isna().sum()
)

display(
    df_dft[["amount_gbp", "payment_amount_gbp"]].head()
)

Invalid or missing amounts: 0


,amount_gbp,payment_amount_gbp
0,"£423,374,000.00",4.233740e+08
1,"£292,000,000.00",2.920000e+08
4,"£153,050,802.26",1.530508e+08
5,"£128,776,527.30",1.287765e+08
6,"£124,200,000.00",1.242000e+08


In [14]:
quality_check = pd.DataFrame({
    "missing_values": df_dft.isna().sum(),
    "blank_strings": df_dft.apply(
        lambda col: col.astype("string").str.strip().eq("").sum()
    ),
    "hash_placeholders": df_dft.apply(
        lambda col: col.astype("string").str.strip().eq("#").sum()
    )
})

display(quality_check)

,missing_values,blank_strings,hash_placeholders
department_family,0,0,0
entity,0,0,0
date,0,0,0
expense_type,0,0,0
expense_area,0,0,0
supplier,0,0,0
transaction_no,0,0,0
item_text,0,0,154
amount_gbp,0,0,0
postal_code,0,0,0


In [15]:
source_columns = df_raw.columns.str.strip().str.lower().str.replace(
    " ", "_", regex=False
).tolist()
source_columns = [
    "amount_gbp" if col == "£" else col
    for col in source_columns
]

exact_duplicates = df_dft.duplicated(
    subset=source_columns,
    keep=False
)

print("Rows in identical-record groups:", exact_duplicates.sum())
print(
    "Repeated transaction references beyond the first:",
    df_dft["transaction_no"].duplicated().sum()
)

if exact_duplicates.any():
    display(df_dft.loc[exact_duplicates].head(10))

Rows in identical-record groups: 8
Repeated transaction references beyond the first: 145


,department_family,entity,date,expense_type,expense_area,supplier,transaction_no,item_text,amount_gbp,postal_code,payment_date,payment_amount_gbp
599,Department for Transport,Department for Transport,15/01/2026,Subsidies Private Se,DG Public Transport and Local Group,KONECTBUS LTD,2000027931,KONECTBUS LTD,"£224,914.25",NR19 1SY,2026-01-15,224914.25
600,Department for Transport,Department for Transport,15/01/2026,Subsidies Private Se,DG Public Transport and Local Group,KONECTBUS LTD,2000027931,KONECTBUS LTD,"£224,914.25",NR19 1SY,2026-01-15,224914.25
1704,Department for Transport,Department for Transport,28/01/2026,IT Consultancy,DG Corporate Delivery Group,XMA Ltd,2000029128,#,"£44,856.79",NG11 7EP,2026-01-28,44856.79
1705,Department for Transport,Department for Transport,28/01/2026,IT Consultancy,DG Corporate Delivery Group,XMA Ltd,2000029128,#,"£44,856.79",NG11 7EP,2026-01-28,44856.79
1900,Department for Transport,Department for Transport,27/01/2026,Training,DG Corporate Delivery Group,KPMG LLP T/A CSL-KPMG LLP,2000028912,#,"£37,777.31",E14 5GL,2026-01-27,37777.31
1901,Department for Transport,Department for Transport,27/01/2026,Training,DG Corporate Delivery Group,KPMG LLP T/A CSL-KPMG LLP,2000028912,#,"£37,777.31",E14 5GL,2026-01-27,37777.31
2320,Department for Transport,Department for Transport,23/01/2026,Research,DG Rail Reform and Strategy Group,Frontier Economics,2000028611,#,"£27,000.00",EC2A 4JE,2026-01-23,27000.00
2321,Department for Transport,Department for Transport,23/01/2026,Research,DG Rail Reform and Strategy Group,Frontier Economics,2000028611,#,"£27,000.00",EC2A 4JE,2026-01-23,27000.00


In [16]:
amounts = df_dft["payment_amount_gbp"]

print("Negative amounts:", (amounts < 0).sum())
print("Zero amounts:", (amounts == 0).sum())
print(
    "Positive amounts of £25,000 or less:",
    ((amounts > 0) & (amounts <= 25_000)).sum()
)

display(
    df_dft.loc[
        amounts <= 25_000,
        ["payment_date", "supplier", "transaction_no",
         "expense_type", "payment_amount_gbp"]
    ].head(10)
)

Negative amounts: 0
Zero amounts: 0
Positive amounts of £25,000 or less: 0


,payment_date,supplier,transaction_no,expense_type,payment_amount_gbp


In [17]:
summary = pd.Series({
    "Payment records": len(df_dft),
    "Distinct supplier names": df_dft["supplier"].nunique(),
    "Published payments (£)": df_dft["payment_amount_gbp"].sum(),
    "Median payment (£)": df_dft["payment_amount_gbp"].median(),
    "Largest payment (£)": df_dft["payment_amount_gbp"].max()
})

display(summary.to_frame("January 2026").style.format("{:,.2f}"))

,January 2026
Payment records,659.00
Distinct supplier names,336.00
Published payments (£),"2,391,570,289.87"
Median payment (£),"165,540.00"
Largest payment (£),"423,374,000.00"


In [18]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

output_path = (
    PROCESSED_DATA_DIR / "dft_central_spending_2026_01_cleaned.csv"
)

df_dft.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Rows saved:", len(df_dft))

Saved to: /Users/dafyddjones/projects/finance-kpi-reporting-pipeline/data/processed/dft_central_spending_2026_01_cleaned.csv
Rows saved: 659
